## Environment check

This cell verifies that the local `nbloss` package is imported from the intended project environment before running the benchmark. The package provides the Smooth Net Benefit training procedure and Net Benefit evaluation functions used later in the notebook.

In [1]:
import nbloss

print("Imported from:", nbloss.__file__)

Imported from: C:\Users\koeng\Downloads\nbloss-main\nbloss-main\nbloss\__init__.py


## Step 1 — TabZilla benchmark datasets

This step defines the OpenML datasets included in the TabZilla logistic regression benchmark. The dataset identifiers are fixed in advance so that the same collection of binary classification tasks can be used consistently across model classes and training objectives.

Datasets are subsequently retrieved directly from OpenML. Additional eligibility checks, including the minimum sample-size requirement and verification that the outcome is binary, are applied when the benchmark is executed.

In [14]:
OPENML_DATASET_IDS = [
    1120,   # Magic telescope
    1053,   # openml__jm1__3904
    4532,   # higgs
    4534,   # phishing websites
    1489,   # phoneme
    1502,   # skin segmentation
    1590,   # adult income
    45072,  # airlines
    151,    # electricity
    4135,   # Amazon_employee_access
    40978,  # internet advertisements
    41434,  # click prediction small
    41150,  # miniBooNE
    40536,  # speeddating
    1043,   # ada agnostic
    1462,   # banknote authentication
    41142,  # christine
    40701,  # churn
    31,     # credit-g
    1471,   # eeg-state
    846,    # elevators
    1038,   # gina agnostic
    821,    # house 16H
    41143,  # jasmine
    1067,   # kc1
    1485,   # madelon
    24,     # mushroom
    1116,   # musk
    1486,   # nomao
    23517,  # numerai28.6
    1487,   # ozone
    1068,   # pc1
    1050,   # pc3
    1049,   # pc4
    41145,  # philippine
    871,    # pollen
    312,    # scene
    38,     # sick
    44,     # spambase
    1570,   # wilt
    45035,  # Albert
    1461,   # bank marketing
    1036,   # sylvia agnostic
    41146,  # sylvine
]

## Step 2 — Cross-validation splits

This step defines the outer train/test splits used throughout the benchmark. Each dataset is partitioned using stratified 5-fold cross-validation with a fixed random seed.

For each of the five runs, one fold is used as the independent test set and the remaining four folds are used for model development. The same fold construction can therefore be used across training objectives and model classes, allowing comparisons to be made on identical observations.

Stratification preserves the class distribution as closely as possible across folds.

In [ ]:

from sklearn.model_selection import StratifiedKFold
import numpy as np



def make_5fold_indices(y: np.ndarray, seed: int):
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=seed,
    )
    return [test_idx for _, test_idx in skf.split(np.zeros_like(y), y)]


def indices_for_run_train_test(folds, run_id: int):
    """
    5-fold rotation:
    - run_id fold = test
    - all other folds = train
    """
    test_fold = run_id % 5

    test_inds = folds[test_fold]

    train_inds = np.concatenate(
        [folds[i] for i in range(5) if i != test_fold],
        axis=0,
    )

    return train_inds, test_inds

## Step 3 — Shared preprocessing

This step defines the preprocessing pipeline applied to each OpenML dataset. Feature types are inferred from the training data and preprocessing transformations are fitted on the outer training fold only before being applied to the corresponding test fold.

Variables are handled as follows:

- binary variables are retained as single binary features;
- string variables and numeric variables with at most 20 unique values are treated as categorical;
- categorical variables are restricted to their 10 most frequent levels, with less frequent levels grouped into an `OTHER` category, and are subsequently one-hot encoded;
- continuous numeric variables are median-imputed and standardized.

Unknown categorical levels in the test data are handled without modifying the training-derived feature space. The resulting model matrices are converted to dense `float32` arrays for use by the PyTorch logistic regression models.

In [40]:

import numpy as np
import pandas as pd

from dataclasses import dataclass
from scipy import sparse
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch.utils.data import TensorDataset, DataLoader


def _to_dense_float32(X):
    if sparse.issparse(X):
        X = X.toarray()
    return np.asarray(X, dtype=np.float32)


def _make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", drop="if_binary", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", drop="if_binary", sparse=False)


def infer_feature_types(X: pd.DataFrame, *, numeric_cat_max_unique: int = 20):
    binary_cols, categorical_cols, numeric_cols = [], [], []

    for c in X.columns:
        s = X[c]
        nunq_including_nan = pd.Series(s).nunique(dropna=False)

        if nunq_including_nan == 2:
            binary_cols.append(c)
            continue

        dtype_name = str(s.dtype)

        if dtype_name in ("object", "category", "string"):
            categorical_cols.append(c)
            continue

        if pd.api.types.is_numeric_dtype(s):
            if pd.Series(s).nunique(dropna=False) <= numeric_cat_max_unique:
                categorical_cols.append(c)
            else:
                numeric_cols.append(c)
            continue

        categorical_cols.append(c)

    return binary_cols, categorical_cols, numeric_cols


class TopKCategoryGrouper(BaseEstimator, TransformerMixin):

    def __init__(self, top_k: int = 10):
        self.top_k = int(top_k)
        self.keep_values_ = None
        self.columns_ = None

    def _as_frame(self, X):
        if isinstance(X, pd.DataFrame):
            return X.copy()
        return pd.DataFrame(X, columns=self.columns_)

    def fit(self, X, y=None):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        self.columns_ = list(X_df.columns)
        self.keep_values_ = {}

        for c in self.columns_:
            s = X_df[c].astype("object")
            s = s.where(~pd.isna(s), "__NaN__").astype(str)
            vc = s.value_counts(dropna=False)
            self.keep_values_[c] = set(vc.head(self.top_k).index.tolist())

        return self

    def transform(self, X):
        X_df = self._as_frame(X)
        out = pd.DataFrame(index=X_df.index)

        for c in self.columns_:
            s = X_df[c].astype("object")
            s = s.where(~pd.isna(s), "__NaN__").astype(str)
            keep = self.keep_values_[c]
            out[c] = s.where(s.isin(keep), "__OTHER__")

        return out

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return np.asarray(self.columns_, dtype=object)
        return np.asarray(input_features, dtype=object)


class BinaryPassthroughEncoder(BaseEstimator, TransformerMixin):

    def __init__(self):
        self.columns_ = None
        self.fill_values_ = {}
        self.value_maps_ = {}

    def _as_frame(self, X):
        if isinstance(X, pd.DataFrame):
            return X.copy()
        return pd.DataFrame(X, columns=self.columns_)

    def fit(self, X, y=None):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        self.columns_ = list(X_df.columns)

        for c in self.columns_:
            s = X_df[c]
            non_missing = s[~pd.isna(s)]

            if len(non_missing) == 0:
                self.fill_values_[c] = 0
                self.value_maps_[c] = {}
                continue

            mode_vals = non_missing.mode(dropna=True)
            fill_val = mode_vals.iloc[0] if len(mode_vals) > 0 else non_missing.iloc[0]
            self.fill_values_[c] = fill_val

            seen = []
            for v in non_missing:
                if v not in seen:
                    seen.append(v)

            if len(seen) == 1:
                mapping = {seen[0]: 0.0}
            else:
                mapping = {seen[0]: 0.0, seen[1]: 1.0}

            self.value_maps_[c] = mapping

        return self

    def transform(self, X):
        X_df = self._as_frame(X)
        out = np.zeros((len(X_df), len(self.columns_)), dtype=np.float32)

        for j, c in enumerate(self.columns_):
            s = X_df[c].copy()
            fill_val = self.fill_values_[c]
            mapping = self.value_maps_[c]

            s = s.where(~pd.isna(s), fill_val)
            default_code = mapping.get(fill_val, 0.0)

            out[:, j] = s.map(lambda v: mapping.get(v, default_code)).astype(np.float32).to_numpy()

        return out

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return np.asarray(self.columns_, dtype=object)
        return np.asarray(input_features, dtype=object)


@dataclass
class PreprocessorBundle:
    binary_cols: list
    categorical_cols: list
    numeric_cols: list
    preprocessor_base: ColumnTransformer
    feature_names: list


def _fit_shared_preprocessor_bundle(
    X_tr: pd.DataFrame,
    *,
    numeric_cat_max_unique: int = 20,
    top_k_categories: int = 10,
):

    binary_cols, categorical_cols, numeric_cols = infer_feature_types(
        X_tr, numeric_cat_max_unique=numeric_cat_max_unique
    )

    cat_pipe = Pipeline(
        steps=[
            ("topk", TopKCategoryGrouper(top_k=top_k_categories)),
            ("ohe", _make_ohe()),
        ]
    )

    num_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    bin_pipe = Pipeline(
        steps=[
            ("binary", BinaryPassthroughEncoder()),
        ]
    )

    preprocessor_base = ColumnTransformer(
        transformers=[
            ("cat", cat_pipe, categorical_cols),
            ("num", num_pipe, numeric_cols),
            ("bin", bin_pipe, binary_cols),
        ],
        remainder="drop",
        sparse_threshold=0.0,
        verbose_feature_names_out=False,
    )

    preprocessor_base.fit(X_tr)

    try:
        feature_names = list(preprocessor_base.get_feature_names_out())
    except Exception:
        feature_names = [f"x{i}" for i in range(preprocessor_base.transform(X_tr.iloc[:1]).shape[1])]

    return PreprocessorBundle(
        binary_cols=binary_cols,
        categorical_cols=categorical_cols,
        numeric_cols=numeric_cols,
        preprocessor_base=preprocessor_base,
        feature_names=feature_names,
    )


def fit_train_preprocessor_and_transform(
    X_tr: pd.DataFrame,
    X_te: pd.DataFrame,
):
    bundle = _fit_shared_preprocessor_bundle(X_tr)

    X_tr_enc = _to_dense_float32(bundle.preprocessor_base.transform(X_tr))
    X_te_enc = _to_dense_float32(bundle.preprocessor_base.transform(X_te))

    if np.isnan(X_tr_enc).any() or np.isnan(X_te_enc).any():
        raise RuntimeError("Preprocessing produced NaNs.")

    return bundle, X_tr_enc, X_te_enc

## Step 4 — Global configuration and logistic regression model

This step defines the global settings used throughout the logistic regression benchmark.

It specifies the computational device and random seeds, batch size, minimum dataset size, L2 regularization grid, Smooth Net Benefit annealing schedule, threshold-grid resolution, and settings for the local calibration procedures.

The decision-relevant threshold interval is defined as a band of ±0.025 around the reference threshold. Smooth Net Benefit training uses inverse-temperature annealing with values 1, 4, and 10.

A single-layer PyTorch logistic regression model is also defined. The model produces logits, which are subsequently used for BCE training, Smooth Net Benefit training, probability prediction, and calibration.

In [ ]:
import random

import numpy as np
import torch
import torch.nn as nn

from nbloss.trainer import set_seed


DEVICE = "cuda" if torch.cuda.is_available() else (
    "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu"
)

SEED_GLOBAL = 1234
MODEL_SEED = 4242

BATCH = 1024
MIN_ROWS = 1000

set_seed(SEED_GLOBAL)

LR_ADAMW = 3e-2
INVERSE_TEMPS = (1.0, 4.0, 10.0)
EPOCHS_PER_TEMP = 300
PATIENCE_HARD = 20
TRAIN_RANGE_POINTS = 11
TEST_RANGE_POINTS = 201

BAND_HALF_WIDTH = 0.025
MID_PREV_LOW, MID_PREV_HIGH = 0.40, 0.60

L2_LAM_GRID = [0.0, 1e-4, 1e-3, 1e-2, 1e-1]

LOCAL_START_HALF_WIDTH = 0.1
LOCAL_EXPAND_STEP = 0.01
LOCAL_MIN_POS = 50
LOCAL_MIN_NEG = 50

TEMP_MAX_ITER = 500
PLATT_MAX_ITER = 500


def band_from_t_ref(t_ref: float, half_width: float = BAND_HALF_WIDTH) -> tuple[float, float]:
    eps = 1e-9
    t_ref = float(np.clip(t_ref, eps, 1.0 - eps))

    t_min = max(eps, t_ref - float(half_width))
    t_max = min(1.0 - eps, t_ref + float(half_width))

    if not t_min < t_max:
        span = min(t_ref - eps, 1.0 - eps - t_ref, float(half_width))
        t_min = t_ref - span
        t_max = t_ref + span

    return float(t_min), float(t_max)


class TorchLR(nn.Module):
    def __init__(self, d_in: int):
        super().__init__()
        self.linear = nn.Linear(d_in, 1, bias=True)

    def forward(self, x):
        return self.linear(x).squeeze(-1)

## Step 5 — OpenML loading, output utilities, and threshold specification

This step defines utility functions for loading benchmark datasets from OpenML and for managing benchmark outputs.

For each OpenML dataset, the predictors and binary target are retrieved together with dataset metadata such as sample size, number of raw features, prevalence, and class labels.

Reference decision thresholds are derived from the prevalence of the outer training fold. The prevalence threshold is evaluated for every dataset. For datasets with prevalence outside the central 40%–60% range, an additional inverse-prevalence threshold is evaluated.

For each reference threshold, Net Benefit is evaluated over a threshold band of ±0.025. Helper functions for reproducible PyTorch data loaders and incremental result storage are also defined in this step.

In [38]:
from pathlib import Path
import numpy as np
import pandas as pd
import openml


# ============================================================================
# Output helpers
# ============================================================================

def _append_csv_safely(df: pd.DataFrame, path: Path):
    header = not path.exists()
    df.to_csv(path, mode="a", header=header, index=False)


def _read_done_registry(path_done: Path) -> set[int]:
    if not path_done.exists():
        return set()

    try:
        df = pd.read_csv(path_done)
        return set(df["openml_id"].astype(int).tolist())
    except Exception:
        return set()


def _mark_dataset_done(openml_id: int, name: str, path_done: Path):
    _append_csv_safely(
        pd.DataFrame([{"openml_id": int(openml_id), "name": str(name)}]),
        path_done,
    )


def make_loader(X, y, batch=BATCH, shuffle=False, seed=SEED_GLOBAL):
    g = torch.Generator().manual_seed(seed)

    ds = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32),
    )

    return DataLoader(
        ds,
        batch_size=batch,
        shuffle=shuffle,
        generator=g,
    )

# ============================================================================
# OpenML loader
# ============================================================================

def load_openml_dataset(did: int):
    ds = openml.datasets.get_dataset(did)

    X_df, y_raw, _, _ = ds.get_data(
        target=ds.default_target_attribute,
        dataset_format="dataframe",
        include_row_id=False,
        include_ignore_attribute=False,
    )

    y_series = pd.Series(y_raw)

    if (
        y_series.dtype.kind in ("U", "S", "O", "b")
        or str(y_series.dtype).startswith("category")
    ):
        y_cat = y_series.astype("category")
        classes = list(y_cat.cat.categories)
        y = y_cat.cat.codes.astype("int64").to_numpy().astype("float32")
    else:
        y = y_series.astype("int64").to_numpy().astype("float32")
        classes = ["0", "1"]

    prev = float((y == 1).mean())

    meta = dict(
        openml_id=int(ds.dataset_id),
        name=str(ds.name),
        n_rows=int(len(y)),
        n_features=int(X_df.shape[1]),
        prevalence=prev,
        pos_label=(classes[1] if len(classes) == 2 else "1"),
        neg_label=(classes[0] if len(classes) == 2 else "0"),
    )

    return ds, X_df, y, meta


# ============================================================================
# Threshold helpers
# ============================================================================

def threshold_specs_from_prevalence(
    prev: float,
    *,
    mid_prev_low: float = 0.40,
    mid_prev_high: float = 0.60,
):
    specs = [
        {
            "threshold_name": "prev",
            "threshold": float(prev),
        }
    ]

    if prev < mid_prev_low or prev > mid_prev_high:
        specs.append(
            {
                "threshold_name": "inverse",
                "threshold": float(1.0 - prev),
            }
        )

    return specs


def band_from_t_ref(
    t_ref: float,
    *,
    half_width: float = 0.025,
):
    eps = 1e-9
    t_min = max(0.0 + eps, float(t_ref) - float(half_width))
    t_max = min(1.0 - eps, float(t_ref) + float(half_width))

    if not (0.0 < t_min < t_max < 1.0):
        mid = min(max(float(t_ref), eps), 1.0 - eps)
        span = min(mid - eps, 1.0 - eps - mid, float(half_width))
        t_min, t_max = mid - span, mid + span

    return float(t_min), float(t_max)

## Step 6 — BCE logistic regression and L2 selection

This step defines the baseline logistic regression training procedure.

Logistic regression is fitted by minimizing binary cross-entropy (BCE) with the LBFGS optimizer. L2 regularization is applied to the model weights but not to the intercept.

For each outer training fold, the L2 penalty is selected from a predefined grid using stratified 5-fold cross-validation within the training data. The value with the lowest mean validation BCE is selected and subsequently used when fitting the baseline model on the full outer training fold.

The selected L2 penalty is also retained during subsequent Smooth Net Benefit training so that the comparison primarily reflects the effect of changing the training objective rather than changing the regularization regime.

In [33]:
from sklearn.model_selection import StratifiedKFold

def l2_penalty_weights_only(model: torch.nn.Module) -> torch.Tensor:
    penalty = torch.zeros((), device=next(model.parameters()).device)

    for name, param in model.named_parameters():
        if param.requires_grad and not name.endswith("bias"):
            penalty = penalty + torch.sum(param ** 2)

    return penalty


@torch.no_grad()
def bce_logits_loss(
    model: torch.nn.Module,
    X_t: torch.Tensor,
    y_t: torch.Tensor,
) -> float:
    model.eval()
    bce = torch.nn.BCEWithLogitsLoss(reduction="mean")
    logits = model(X_t).view(-1)
    y_t = y_t.float().view(-1)
    return float(bce(logits, y_t).detach().cpu().item())

def fit_bce_lbfgs(
    make_model_fn,
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    *,
    l2_lambda: float,
    max_iter: int = 500,
    tol_grad: float = 1e-7,
    tol_change: float = 1e-9,
    history_size: int = 100,
    device: str = "cpu",
):
    device_t = torch.device(device)

    X = torch.tensor(X_tr, dtype=torch.float32, device=device_t)
    y = torch.tensor(y_tr, dtype=torch.float32, device=device_t).view(-1)

    model = make_model_fn().to(device_t)
    bce = nn.BCEWithLogitsLoss(reduction="mean")

    opt = torch.optim.LBFGS(
        model.parameters(),
        lr=1.0,
        max_iter=int(max_iter),
        tolerance_grad=float(tol_grad),
        tolerance_change=float(tol_change),
        history_size=int(history_size),
        line_search_fn="strong_wolfe",
    )

    l2_lambda_t = torch.tensor(float(l2_lambda), device=device_t)

    def closure():
        opt.zero_grad(set_to_none=True)

        logits = model(X).view(-1)
        loss = bce(logits, y) + l2_lambda_t * l2_penalty_weights_only(model)

        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite LBFGS loss: {loss.detach().item()}")

        loss.backward()
        return loss

    opt.step(closure)

    train_bce = bce_logits_loss(model, X, y)

    return model, train_bce


def select_l2_by_cv_bce(
    lambdas,
    *,
    make_model_fn,
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    device: str = "cpu",
    n_splits: int = 5,
    lbfgs_max_iter: int = 500,
    seed: int = 1234,
):
    skf = StratifiedKFold(
        n_splits=int(n_splits),
        shuffle=True,
        random_state=int(seed),
    )

    rows = []
    best_lambda = None
    best_cv_bce = float("inf")

    for lam in lambdas:
        fold_bces = []

        for fold_id, (tr_idx, va_idx) in enumerate(skf.split(X_tr, y_tr)):
            X_tr_f = X_tr[tr_idx]
            y_tr_f = y_tr[tr_idx]
            X_va_f = X_tr[va_idx]
            y_va_f = y_tr[va_idx]

            model, _ = fit_bce_lbfgs(
                make_model_fn,
                X_tr_f,
                y_tr_f,
                l2_lambda=float(lam),
                max_iter=int(lbfgs_max_iter),
                device=device,
            )

            device_t = torch.device(device)
            X_va_t = torch.tensor(X_va_f, dtype=torch.float32, device=device_t)
            y_va_t = torch.tensor(y_va_f, dtype=torch.float32, device=device_t).view(-1)

            fold_bce = bce_logits_loss(model, X_va_t, y_va_t)
            fold_bces.append(fold_bce)

        mean_bce = float(np.mean(fold_bces))

        rows.append(
            {
                "l2_lambda": float(lam),
                "cv_bce_mean": mean_bce,
                "cv_bce_folds": fold_bces,
            }
        )

        if mean_bce < best_cv_bce:
            best_lambda = float(lam)
            best_cv_bce = mean_bce

    return best_lambda, best_cv_bce, rows

## Step 7 — Smooth Net Benefit training and Net Benefit metric

This step imports the Smooth Net Benefit training procedure and the Net Benefit evaluation function from the `nbloss` package.

Smooth Net Benefit training starts from the BCE-trained logistic regression model and continues optimization using the differentiable Smooth Net Benefit objective. Training uses inverse-temperature annealing while retaining the L2 penalty selected for the corresponding BCE model.

Model selection during Smooth Net Benefit training is based on Net Benefit over the decision-relevant threshold band.

The imported Net Benefit metric is also used for final evaluation on the independent outer test fold.

In [34]:
from nbloss.trainer import nb_anneal_only_with_l2
from nbloss.metrics import average_nb_over_range

## Step 8 — Prediction, local calibration, and Net Benefit evaluation

This step defines the prediction, calibration, and decision-analytic evaluation functions used after model fitting.

Model logits are converted to predicted probabilities using the sigmoid function. Net Benefit is evaluated by averaging across a dense grid of thresholds within the predefined decision-relevant threshold band.

Two post-hoc local calibration methods are implemented as comparators: local temperature scaling and local Platt scaling. For each reference threshold, a local subset of the training data is selected around that threshold. The interval is expanded when necessary until it contains sufficient positive and negative observations.

Temperature scaling estimates a single multiplicative temperature parameter for the logits, whereas Platt scaling estimates both a slope and an intercept. Calibration parameters are estimated using training data only and are then applied to predictions in the independent test fold.

In [35]:
def sigmoid_np(logits_np: np.ndarray) -> np.ndarray:
    logits_t = torch.tensor(np.asarray(logits_np, dtype=np.float32).reshape(-1))
    return torch.sigmoid(logits_t).detach().cpu().numpy().astype(np.float64)


@torch.no_grad()
def predict_logits_torch(
    model: nn.Module,
    X_np: np.ndarray,
    *,
    device: str = DEVICE,
) -> np.ndarray:
    device_t = torch.device(device)
    model.eval()
    X_t = torch.tensor(X_np, dtype=torch.float32, device=device_t)
    return model(X_t).detach().cpu().view(-1).numpy().astype(np.float64)


def subset_counts(mask, y_np):
    y_sub = np.asarray(y_np).reshape(-1)[np.asarray(mask, dtype=bool)]
    n_pos = int(np.sum(y_sub == 1))
    n_neg = int(np.sum(y_sub == 0))
    return int(len(y_sub)), n_pos, n_neg


def find_local_calibration_range(
    probs_train: np.ndarray,
    y_train: np.ndarray,
    *,
    t_ref: float,
    start_half_width: float = LOCAL_START_HALF_WIDTH,
    expand_step: float = LOCAL_EXPAND_STEP,
    min_pos: int = LOCAL_MIN_POS,
    min_neg: int = LOCAL_MIN_NEG,
) -> dict:
    probs_train = np.asarray(probs_train, dtype=float).reshape(-1)
    y_train = np.asarray(y_train).reshape(-1)

    half_width = float(start_half_width)
    n_expand_steps = 0

    while True:
        low = max(0.0, float(t_ref) - half_width)
        high = min(1.0, float(t_ref) + half_width)

        mask = (probs_train >= low) & (probs_train <= high)
        n, n_pos, n_neg = subset_counts(mask, y_train)

        met_minimum = (n_pos >= int(min_pos)) and (n_neg >= int(min_neg))
        used_full_range = (low <= 0.0) and (high >= 1.0)

        if met_minimum or used_full_range:
            return {
                "low": float(low),
                "high": float(high),
                "half_width": float(half_width),
                "range_width": float(high - low),
                "n_expand_steps": int(n_expand_steps),
                "n": int(n),
                "n_pos": int(n_pos),
                "n_neg": int(n_neg),
                "met_minimum": bool(met_minimum),
                "used_full_range": bool(used_full_range),
                "mask": mask,
            }

        half_width += float(expand_step)
        n_expand_steps += 1


def fit_temperature_from_logits_np(
    logits_np: np.ndarray,
    y_np: np.ndarray,
    *,
    max_iter: int = TEMP_MAX_ITER,
    device: str = DEVICE,
) -> dict:
    device_t = torch.device(device)

    logits = torch.tensor(
        np.asarray(logits_np, dtype=np.float32).reshape(-1),
        device=device_t,
    )
    y = torch.tensor(
        np.asarray(y_np, dtype=np.float32).reshape(-1),
        device=device_t,
    )

    bce = nn.BCEWithLogitsLoss(reduction="mean")

    with torch.no_grad():
        nll_before = float(bce(logits, y).detach().cpu().item())

    log_temperature = torch.zeros(
        (),
        dtype=torch.float32,
        device=device_t,
        requires_grad=True,
    )

    optimizer = torch.optim.LBFGS(
        [log_temperature],
        lr=0.1,
        max_iter=int(max_iter),
        line_search_fn="strong_wolfe",
    )

    def closure():
        optimizer.zero_grad(set_to_none=True)
        temperature = torch.exp(log_temperature).clamp(min=1e-6, max=1e6)
        loss = bce(logits / temperature, y)

        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite temperature loss: {loss.detach().item()}")

        loss.backward()
        return loss

    optimizer.step(closure)

    with torch.no_grad():
        temperature = torch.exp(log_temperature).clamp(min=1e-6, max=1e6)
        nll_after = float(bce(logits / temperature, y).detach().cpu().item())

    return {
        "temperature": float(temperature.detach().cpu().item()),
        "nll_before": nll_before,
        "nll_after": nll_after,
    }


def apply_local_temperature_to_logits(
    logits_np: np.ndarray,
    probs_reference_np: np.ndarray,
    *,
    low: float,
    high: float,
    temperature: float,
) -> tuple[np.ndarray, np.ndarray]:
    logits_out = np.asarray(logits_np, dtype=np.float64).reshape(-1).copy()
    probs_reference_np = np.asarray(probs_reference_np, dtype=float).reshape(-1)

    mask = (probs_reference_np >= float(low)) & (probs_reference_np <= float(high))
    logits_out[mask] = logits_out[mask] / float(temperature)

    return logits_out, mask


def fit_platt_from_logits_np(
    logits_np: np.ndarray,
    y_np: np.ndarray,
    *,
    max_iter: int = PLATT_MAX_ITER,
    device: str = DEVICE,
) -> dict:
    device_t = torch.device(device)

    logits = torch.tensor(
        np.asarray(logits_np, dtype=np.float32).reshape(-1),
        device=device_t,
    )
    y = torch.tensor(
        np.asarray(y_np, dtype=np.float32).reshape(-1),
        device=device_t,
    )

    bce = nn.BCEWithLogitsLoss(reduction="mean")

    with torch.no_grad():
        nll_before = float(bce(logits, y).detach().cpu().item())

    slope = torch.ones((), dtype=torch.float32, device=device_t, requires_grad=True)
    intercept = torch.zeros((), dtype=torch.float32, device=device_t, requires_grad=True)

    optimizer = torch.optim.LBFGS(
        [slope, intercept],
        lr=0.1,
        max_iter=int(max_iter),
        line_search_fn="strong_wolfe",
    )

    def closure():
        optimizer.zero_grad(set_to_none=True)
        calibrated_logits = slope * logits + intercept
        loss = bce(calibrated_logits, y)

        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite Platt loss: {loss.detach().item()}")

        loss.backward()
        return loss

    optimizer.step(closure)

    with torch.no_grad():
        calibrated_logits = slope * logits + intercept
        nll_after = float(bce(calibrated_logits, y).detach().cpu().item())

    return {
        "platt_slope": float(slope.detach().cpu().item()),
        "platt_intercept": float(intercept.detach().cpu().item()),
        "nll_before": nll_before,
        "nll_after": nll_after,
    }


def apply_local_platt_to_logits(
    logits_np: np.ndarray,
    probs_reference_np: np.ndarray,
    *,
    low: float,
    high: float,
    slope: float,
    intercept: float,
) -> tuple[np.ndarray, np.ndarray]:
    logits_out = np.asarray(logits_np, dtype=np.float64).reshape(-1).copy()
    probs_reference_np = np.asarray(probs_reference_np, dtype=float).reshape(-1)

    mask = (probs_reference_np >= float(low)) & (probs_reference_np <= float(high))
    logits_out[mask] = float(slope) * logits_out[mask] + float(intercept)

    return logits_out, mask


def evaluate_nb_from_logits_np(
    logits_np: np.ndarray,
    y_np: np.ndarray,
    *,
    thresh_min: float,
    thresh_max: float,
    num_points: int = TEST_RANGE_POINTS,
) -> float:
    logits_t = torch.tensor(np.asarray(logits_np, dtype=np.float32).reshape(-1))
    y_t = torch.tensor(np.asarray(y_np, dtype=np.float32).reshape(-1))

    return float(
        average_nb_over_range(
            logits_t,
            y_t,
            thresh_min=float(thresh_min),
            thresh_max=float(thresh_max),
            num_points=int(num_points),
            input_is_logit=True,
            method="mean",
        )
    )

## Step 9 — OpenML logistic regression benchmark

This step executes the complete logistic regression benchmark across the predefined OpenML datasets.

For each eligible dataset, five outer train/test rotations are evaluated. Within each outer training fold, preprocessing is fitted, the L2 penalty is selected, and a BCE-trained logistic regression model is fitted. The BCE model is then used as the initialization for Smooth Net Benefit training.

For each reference decision threshold, the following approaches are evaluated on the same independent test fold:

- BCE-trained logistic regression;
- BCE-trained logistic regression with local temperature scaling;
- BCE-trained logistic regression with local Platt scaling;
- Smooth Net Benefit-trained logistic regression;
- Smooth Net Benefit-trained logistic regression with local temperature scaling;
- Smooth Net Benefit-trained logistic regression with local Platt scaling.

Net Benefit is evaluated over the threshold band associated with each reference threshold. The benchmark stores dataset characteristics, fold information, selected regularization strength, calibration diagnostics, test Net Benefit, and differences relative to the BCE baseline.

Results are accumulated across datasets, thresholds, folds, and model variants for subsequent statistical analysis.


In [41]:
from nbloss.trainer import nb_anneal_only_with_l2

results = []


def add_result_row(
    *,
    openml_id: int,
    dataset_name: str,
    run_id: int,
    threshold_name: str,
    model_name: str,
    prev_train: float,
    prev_test: float,
    t_ref: float,
    t_min: float,
    t_max: float,
    l2_lambda: float,
    cv_bce: float,
    train_bce: float,
    test_nb: float,
    delta_vs_bce: float,
    n_rows: int,
    n_features_raw: int,
    n_features_encoded: int,
    pos_label,
    neg_label,
    local_info: dict | None = None,
    calibration_info: dict | None = None,
):
    local_info = local_info or {}
    calibration_info = calibration_info or {}

    row = {
        "openml_id": int(openml_id),
        "dataset": str(dataset_name),
        "run_id": int(run_id),
        "threshold_name": str(threshold_name),
        "model": str(model_name),
        "prev_train": float(prev_train),
        "prev_test": float(prev_test),
        "t_ref": float(t_ref),
        "t_min": float(t_min),
        "t_max": float(t_max),
        "l2_lambda": float(l2_lambda),
        "cv_bce": float(cv_bce),
        "train_bce": float(train_bce),
        "test_nb": float(test_nb),
        "delta_vs_bce": float(delta_vs_bce),
        "local_low": local_info.get("low", np.nan),
        "local_high": local_info.get("high", np.nan),
        "local_half_width": local_info.get("half_width", np.nan),
        "local_range_width": local_info.get("range_width", np.nan),
        "local_expand_steps": local_info.get("n_expand_steps", np.nan),
        "local_train_n": local_info.get("n", np.nan),
        "local_train_pos": local_info.get("n_pos", np.nan),
        "local_train_neg": local_info.get("n_neg", np.nan),
        "local_met_minimum": local_info.get("met_minimum", np.nan),
        "local_used_full_range": local_info.get("used_full_range", np.nan),
        "temperature": calibration_info.get("temperature", np.nan),
        "platt_slope": calibration_info.get("platt_slope", np.nan),
        "platt_intercept": calibration_info.get("platt_intercept", np.nan),
        "train_local_nll_before": calibration_info.get("nll_before", np.nan),
        "train_local_nll_after": calibration_info.get("nll_after", np.nan),
        "n_rows": int(n_rows),
        "n_features_raw": int(n_features_raw),
        "n_features_encoded": int(n_features_encoded),
        "pos_label": pos_label,
        "neg_label": neg_label,
    }

    results.append(row)
    print(pd.DataFrame([row]))


for openml_id in OPENML_DATASET_IDS:
    print(f"\n==================== OpenML dataset {openml_id} ====================")

    try:
        ds, X_df, y, meta = load_openml_dataset(openml_id)
    except Exception as e:
        print(f"[ERROR] Could not load OpenML dataset {openml_id}: {e}")
        continue

    dataset_name = meta["name"]

    if int(meta["n_rows"]) < int(MIN_ROWS):
        print(
            f"[SKIP] {dataset_name} ({openml_id}) has only "
            f"{meta['n_rows']} rows; MIN_ROWS={MIN_ROWS}."
        )
        continue

    if len(np.unique(y)) != 2:
        print(f"[SKIP] {dataset_name} ({openml_id}) is not binary after loading.")
        continue

    folds = make_5fold_indices(y, seed=SEED_GLOBAL)

    print(
        f"[DATASET] {dataset_name} | "
        f"n={meta['n_rows']} | "
        f"raw_features={meta['n_features']} | "
        f"prevalence={meta['prevalence']:.4f}"
    )

    for run_id in range(5):
        split_seed = int(SEED_GLOBAL) + int(run_id)

        train_idx, test_idx = indices_for_run_train_test(folds, run_id)

        X_tr_df = X_df.iloc[train_idx].copy()
        X_te_df = X_df.iloc[test_idx].copy()

        ytr = np.asarray(y[train_idx], dtype=np.float32).reshape(-1)
        yte = np.asarray(y[test_idx], dtype=np.float32).reshape(-1)

        prev_train = float(ytr.mean())
        prev_test = float(yte.mean())

        bundle, Xtr, Xte = fit_train_preprocessor_and_transform(
            X_tr_df,
            X_te_df,
        )

        Xtr = np.asarray(Xtr, dtype=np.float32)
        Xte = np.asarray(Xte, dtype=np.float32)

        d_in = int(Xtr.shape[1])

        def make_lr_model():
            set_seed(MODEL_SEED)
            return TorchLR(d_in)

        lam_star, best_cv_bce, cv_grid = select_l2_by_cv_bce(
            L2_LAM_GRID,
            make_model_fn=make_lr_model,
            X_tr=Xtr,
            y_tr=ytr,
            device=DEVICE,
            n_splits=5,
            lbfgs_max_iter=500,
            seed=split_seed,
        )

        print(
            f"\n[{dataset_name} | run {run_id}] "
            f"prev_train={prev_train:.4f} | "
            f"prev_test={prev_test:.4f} | "
            f"encoded_features={d_in} | "
            f"selected l2_lambda={lam_star:g} | "
            f"best inner-CV BCE={best_cv_bce:.6f}"
        )

        train_dl = make_loader(
            Xtr,
            ytr,
            batch=BATCH,
            shuffle=True,
            seed=split_seed,
        )

        threshold_specs = threshold_specs_from_prevalence(prev_train)

        for spec in threshold_specs:
            threshold_name = spec["threshold_name"]
            t_ref = float(spec["threshold"])
            t_min, t_max = band_from_t_ref(t_ref)

            print(
                f"\n[{dataset_name} | run {run_id}] "
                f"=== THRESHOLD: {threshold_name} "
                f"(t_ref={t_ref:.4f}, [{t_min:.4f}, {t_max:.4f}]) ==="
            )

            bce_model, train_bce = fit_bce_lbfgs(
                make_lr_model,
                Xtr,
                ytr,
                l2_lambda=float(lam_star),
                max_iter=500,
                device=DEVICE,
            )

            logits_bce_train = predict_logits_torch(bce_model, Xtr, device=DEVICE)
            logits_bce_test = predict_logits_torch(bce_model, Xte, device=DEVICE)

            probs_bce_train = sigmoid_np(logits_bce_train)
            probs_bce_test = sigmoid_np(logits_bce_test)

            nb_bce = evaluate_nb_from_logits_np(
                logits_bce_test,
                yte,
                thresh_min=t_min,
                thresh_max=t_max,
                num_points=TEST_RANGE_POINTS,
            )

            print(f"[TEST][{threshold_name}] BCE-LR NB={nb_bce:.6f}")

            local_bce = find_local_calibration_range(
                probs_bce_train,
                ytr,
                t_ref=float(t_ref),
                start_half_width=LOCAL_START_HALF_WIDTH,
                expand_step=LOCAL_EXPAND_STEP,
                min_pos=LOCAL_MIN_POS,
                min_neg=LOCAL_MIN_NEG,
            )

            logits_bce_train_local = logits_bce_train[local_bce["mask"]]
            ytr_bce_local = ytr[local_bce["mask"]]

            temp_bce_fit = fit_temperature_from_logits_np(
                logits_bce_train_local,
                ytr_bce_local,
                max_iter=TEMP_MAX_ITER,
                device=DEVICE,
            )

            platt_bce_fit = fit_platt_from_logits_np(
                logits_bce_train_local,
                ytr_bce_local,
                max_iter=PLATT_MAX_ITER,
                device=DEVICE,
            )

            logits_bce_temp_test, _ = apply_local_temperature_to_logits(
                logits_bce_test,
                probs_bce_test,
                low=local_bce["low"],
                high=local_bce["high"],
                temperature=temp_bce_fit["temperature"],
            )

            logits_bce_platt_test, _ = apply_local_platt_to_logits(
                logits_bce_test,
                probs_bce_test,
                low=local_bce["low"],
                high=local_bce["high"],
                slope=platt_bce_fit["platt_slope"],
                intercept=platt_bce_fit["platt_intercept"],
            )

            nb_bce_temp = evaluate_nb_from_logits_np(
                logits_bce_temp_test,
                yte,
                thresh_min=t_min,
                thresh_max=t_max,
                num_points=TEST_RANGE_POINTS,
            )

            nb_bce_platt = evaluate_nb_from_logits_np(
                logits_bce_platt_test,
                yte,
                thresh_min=t_min,
                thresh_max=t_max,
                num_points=TEST_RANGE_POINTS,
            )

            snb_start = TorchLR(d_in).to(DEVICE)
            snb_start.load_state_dict(
                {
                    k: v.detach().cpu().clone()
                    for k, v in bce_model.state_dict().items()
                }
            )

            snb_model = nb_anneal_only_with_l2(
                snb_start,
                train_dl,
                thresh_min=float(t_min),
                thresh_max=float(t_max),
                num_points_train=int(TRAIN_RANGE_POINTS),
                inverse_temps=tuple(INVERSE_TEMPS),
                epochs_per_temp=int(EPOCHS_PER_TEMP),
                patience_hard=int(PATIENCE_HARD),
                hard_range_num_points=int(TEST_RANGE_POINTS),
                lr_adam=float(LR_ADAMW),
                l2_lambda=float(lam_star),
                penalty_fn=l2_penalty_weights_only,
                device=DEVICE,
                seed=int(MODEL_SEED) + 1000 * run_id + 17,
                log_every=20,
            )

            logits_snb_test = predict_logits_torch(snb_model, Xte, device=DEVICE)

            nb_snb = evaluate_nb_from_logits_np(
                logits_snb_test,
                yte,
                thresh_min=t_min,
                thresh_max=t_max,
                num_points=TEST_RANGE_POINTS,
            )

            print(
                f"[TEST][{threshold_name}] "
                f"SNB-LR NB={nb_snb:.6f} | Δ={nb_snb - nb_bce:+.6f}"
            )

            add_result_row(
                openml_id=openml_id,
                dataset_name=dataset_name,
                run_id=run_id,
                threshold_name=threshold_name,
                model_name="bce_lr",
                prev_train=prev_train,
                prev_test=prev_test,
                t_ref=t_ref,
                t_min=t_min,
                t_max=t_max,
                l2_lambda=lam_star,
                cv_bce=best_cv_bce,
                train_bce=train_bce,
                test_nb=nb_bce,
                delta_vs_bce=0.0,
                n_rows=meta["n_rows"],
                n_features_raw=meta["n_features"],
                n_features_encoded=d_in,
                pos_label=meta["pos_label"],
                neg_label=meta["neg_label"],
            )

            add_result_row(
                openml_id=openml_id,
                dataset_name=dataset_name,
                run_id=run_id,
                threshold_name=threshold_name,
                model_name="bce_lr_local_temperature",
                prev_train=prev_train,
                prev_test=prev_test,
                t_ref=t_ref,
                t_min=t_min,
                t_max=t_max,
                l2_lambda=lam_star,
                cv_bce=best_cv_bce,
                train_bce=train_bce,
                test_nb=nb_bce_temp,
                delta_vs_bce=nb_bce_temp - nb_bce,
                n_rows=meta["n_rows"],
                n_features_raw=meta["n_features"],
                n_features_encoded=d_in,
                pos_label=meta["pos_label"],
                neg_label=meta["neg_label"],
                local_info=local_bce,
                calibration_info=temp_bce_fit,
            )

            add_result_row(
                openml_id=openml_id,
                dataset_name=dataset_name,
                run_id=run_id,
                threshold_name=threshold_name,
                model_name="bce_lr_local_platt",
                prev_train=prev_train,
                prev_test=prev_test,
                t_ref=t_ref,
                t_min=t_min,
                t_max=t_max,
                l2_lambda=lam_star,
                cv_bce=best_cv_bce,
                train_bce=train_bce,
                test_nb=nb_bce_platt,
                delta_vs_bce=nb_bce_platt - nb_bce,
                n_rows=meta["n_rows"],
                n_features_raw=meta["n_features"],
                n_features_encoded=d_in,
                pos_label=meta["pos_label"],
                neg_label=meta["neg_label"],
                local_info=local_bce,
                calibration_info=platt_bce_fit,
            )

            add_result_row(
                openml_id=openml_id,
                dataset_name=dataset_name,
                run_id=run_id,
                threshold_name=threshold_name,
                model_name="snb_lr",
                prev_train=prev_train,
                prev_test=prev_test,
                t_ref=t_ref,
                t_min=t_min,
                t_max=t_max,
                l2_lambda=lam_star,
                cv_bce=best_cv_bce,
                train_bce=train_bce,
                test_nb=nb_snb,
                delta_vs_bce=nb_snb - nb_bce,
                n_rows=meta["n_rows"],
                n_features_raw=meta["n_features"],
                n_features_encoded=d_in,
                pos_label=meta["pos_label"],
                neg_label=meta["neg_label"],
            )

            print(
                f"[CAL][{threshold_name}] "
                f"BCE-temp Δ={nb_bce_temp - nb_bce:+.6f} | "
                f"BCE-Platt Δ={nb_bce_platt - nb_bce:+.6f} | "
                f"SNB Δ={nb_snb - nb_bce:+.6f}"
            )

        if DEVICE == "cuda":
            torch.cuda.empty_cache()


results_df = pd.DataFrame(results)

print("\nFinished.")
print("results_df shape:", results_df.shape)

try:
    display(results_df.head())
except NameError:
    print(results_df.head())


==================== OpenML dataset 1120 ====================
[DATASET] MagicTelescope | n=19020 | raw_features=10 | prevalence=0.3516

[MagicTelescope | run 0] prev_train=0.3517 | prev_test=0.3515 | encoded_features=10 | selected l2_lambda=0.0001 | best inner-CV BCE=0.460241

[MagicTelescope | run 0] === THRESHOLD: prev (t_ref=0.3517, [0.3267, 0.3767]) ===
[TEST][prev] BCE-LR NB=0.189467
[SNB start] initial train hard NB range = 0.188410
[SNB inverse_temp=1] epoch 020 | train_loss=-0.180577 | train_hard_nb=0.194456
[SNB inverse_temp=1] epoch 040 | train_loss=-0.182000 | train_hard_nb=0.195302
[SNB inverse_temp=1] epoch 060 | train_loss=-0.182075 | train_hard_nb=0.195501
>>> Early stop inverse-temperature phase.
[commit] inverse_temp=1 improved global train hard NB: 0.188410 → 0.195603
[SNB inverse_temp=4] epoch 020 | train_loss=-0.192485 | train_hard_nb=0.196872
>>> Early stop inverse-temperature phase.
[commit] inverse_temp=4 improved global train hard NB: 0.195603 → 0.197161
[SNB i

KeyboardInterrupt: 